# 10 — Pre-Flight Check Every Prompt Before It Costs You Tokens

**The pitch.** Most LLM pipelines are blind: write prompt, send it, hope. When the output is bad, you don't know if the prompt was weak or the model was confused. mycontext has three measurement tools: `QualityMetrics` scores your prompt *before* execution. `OutputEvaluator` scores the output *after*. `ContextAmplificationIndex` measures how much better a templated prompt performs vs. a raw one — in hard numbers.

**Who this is for:** Anyone who cares about cost efficiency, result quality, or has to justify prompt choices to a team.

**What you'll learn:**
- `QualityMetrics` — prompt quality scoring (heuristic / LLM / hybrid)
- `OutputEvaluator` — output quality scoring across 7 dimensions
- `ContextAmplificationIndex` — template lift: raw vs. templated, hard number
- `TemplateBenchmark` — systematic benchmark runs from YAML definitions
- How to wire quality gates into a pipeline (block if score is too low)
- How **Constraints** quality fields affect `QualityMetrics`, and how **`forbidden_phrases`** ties into `OutputEvaluator` (0.11+; research-flow **`assemble()`** includes quality copy under **GUARD RAILS** from **0.11.1**)

**Visual map** → [assets/overview.html](assets/overview.html)

**Next:** **11** = Multi-template pipeline. **06** = Full intelligence reference.

## Research grounding

> **Pre-execution quality signals:** Structural features of prompts — rule specificity, output contract presence, grounding protocol, recency-zone task placement — correlate strongly with output quality *(Zhou et al. 2022 — "Large Language Models Are Human-Level Prompt Engineers")*. `QualityMetrics(mode="heuristic")` operationalises these structural features as a free, <1ms scoring signal — no LLM call needed.

> **CAI methodology:** The Context Amplification Index runs two parallel LLM calls (templated vs. raw prompt on the same question), scores both outputs with `OutputEvaluator`, and reports the ratio. Average CAI across mycontext's 200+ analytical test cases: **1.8–2.3x**.

> **Output evaluation dimensions:** The 7 dimensions in `OutputEvaluator` (groundedness, relevance, completeness, structure, register fit, specificity, evidence citation) are adapted from the RAGAS evaluation framework and extended for general-purpose LLM output assessment.

> **0.11+ quality hooks:** `QualityMetrics` gives extra credit when `constraints.self_check`, `constraints.verbosity`, or `constraints.forbidden_phrases` are populated — the prompt explicitly steers verification, detail level, and banned wording. **0.11.1** also emits those paragraphs inside **## GUARD RAILS** when you `assemble()` with `research_flow=True`. After execution, `OutputEvaluator` penalizes answers that contain any phrase listed in `constraints.forbidden_phrases`.

## The three measurement layers

```
  [Prompt]  ──── QualityMetrics ────► score/100  (before LLM call)
      │
      ▼
  [LLM call]
      │
  [Output]  ──── OutputEvaluator ───► score/100  (after LLM call)

  CAI = output_with_template / output_without_template  (lift ratio)
```

| Tool | When | Needs LLM? | Key metric |
|------|------|------------|------------|
| `QualityMetrics` | Before call | Optional | Prompt score 0–100 |
| `OutputEvaluator` | After call | Optional | Output score 0–100 |
| `ContextAmplificationIndex` | Anytime | Yes (calls LLM twice) | CAI ratio |
| `TemplateBenchmark` | Batch testing | Yes | Aggregate CAI |

## Install and setup

In [1]:
# %pip install -q -U "mycontext-ai>=0.11.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)

import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


**mycontext-ai** `0.11.0`

## Step 1 — Set up three prompts to compare

We'll measure the same question across three prompt quality levels: raw (bad), improved (better), and templated (best).

In [2]:
from mycontext import Context, Guidance, Directive, Constraints
from mycontext.intelligence import transform, get_pattern_class

question = "What are the key risks of deploying an ML model in a regulated financial environment?"

# Level 1: Raw (what most people ship)
raw_ctx = Context(
    guidance=Guidance(role="AI assistant", rules=["Be helpful"]),
    directive=Directive(content=question),
)

# Level 2: Manually improved (better but still hand-crafted)
improved_ctx = Context(
    guidance=Guidance(
        role="Senior ML risk analyst with 10 years in regulated financial environments",
        rules=[
            "Must cover model risk, data risk, regulatory risk, and operational risk",
            "Must cite specific frameworks (SR 11-7, EBA guidelines) where relevant",
            "Must not give generic advice — ground every point in financial ML specifics",
        ],
        style="Professional, structured, evidence-based",
    ),
    directive=Directive(content=question),
    constraints=Constraints(
        format_rules=["Use numbered sections", "Each risk must include mitigation"],
    ),
)

# Level 3: Template-driven (let mycontext do the heavy lifting)
templated_ctx = transform(question)

template_name = templated_ctx.metadata.get("template", "auto-selected")
md(f"Template selected by transform(): {template_name}")


Template selected by transform(): auto-selected

## Step 2 — `QualityMetrics`: heuristic scoring (no API key)

From **0.11.0**, populated **`Constraints`** quality fields (`self_check`, `verbosity`, `forbidden_phrases`, plus related posture flags) increase the heuristic score — they signal a prompt that constrains how the model answers, not only what it must cover. With **0.11.1+**, the same text is present in the assembled **GUARD RAILS** section for research-flow contexts, so the heuristic sees it in the prompt string.

In [3]:
from mycontext.intelligence import QualityMetrics

qm = QualityMetrics(mode="heuristic")

raw_score = qm.evaluate(raw_ctx)
improved_score = qm.evaluate(improved_ctx)
templated_score = qm.evaluate(templated_ctx)

md(f"{'Prompt':<35} {'Score':>8} {'Verdict'}")
md("-" * 60)
md(f"{'Raw (generic assistant)':<35} {raw_score.overall * 100:>7.0f}/100  {'✗ Weak' if raw_score.overall < 0.5 else '~ Okay'}")
md(f"{'Manually improved':<35} {improved_score.overall * 100:>7.0f}/100  {'✓ Good' if improved_score.overall >= 0.65 else '~ Okay'}")
md(f"{'Template-driven (transform)':<35} {templated_score.overall * 100:>7.0f}/100  {'✓ Strong' if templated_score.overall >= 0.75 else '✓ Good'}")


Prompt                                 Score Verdict

------------------------------------------------------------

Raw (generic assistant)                  13/100  ✗ Weak

Manually improved                        74/100  ✓ Good

Template-driven (transform)              84/100  ✓ Strong

In [4]:
# Dimension breakdown for the templated prompt
md("Quality dimension breakdown — templated prompt:")
for _dk, _dv in templated_score.dimensions.items():
    _pct = _dv * 100.0
    _filled = int(_pct / 10)
    _bar = '█' * _filled + '░' * (10 - _filled)
    md(f"  {_dk.name.title():<28} {_bar} {_pct:.0f}/100")


Quality dimension breakdown — templated prompt:

  Clarity                      ██████████ 100/100

  Completeness                 ████████░░ 85/100

  Specificity                  ██████░░░░ 65/100

  Relevance                    ███████░░░ 70/100

  Structure                    ██████████ 100/100

  Efficiency                   ███████░░░ 70/100

## Step 3 — Wire a quality gate into your pipeline

Block LLM execution if the prompt quality is too low. This pattern catches bad prompts before they waste tokens.

In [5]:
QUALITY_THRESHOLD = 0.60  # SDK uses 0–1 (≈60/100)

def gated_execute(ctx, threshold=QUALITY_THRESHOLD, provider=None):
    """Execute context only if prompt quality meets the threshold."""
    qm = QualityMetrics(mode="heuristic")
    score = qm.evaluate(ctx)

    if score.overall < threshold:
        return None, {
            "gated": True,
            "reason": f"Prompt quality {score.overall * 100:.0f}/100 is below threshold {threshold * 100:.0f}/100",
            "score": score,
        }

    if provider:
        response = ctx.execute(provider=provider, user=ctx.directive.content if ctx.directive else "")
        return response, {"gated": False, "score": score}
    else:
        return "[LLM call would happen here]", {"gated": False, "score": score}

# Try the raw prompt — should be blocked
result, meta = gated_execute(raw_ctx, threshold=0.60)
md(f"Raw prompt → {'BLOCKED' if meta['gated'] else 'PASSED'}")
md(f"  Reason: {meta.get('reason', 'passed quality gate')}")

# Try the templated prompt — should pass
result, meta = gated_execute(templated_ctx, threshold=0.60)
md(f"\nTemplated prompt → {'BLOCKED' if meta['gated'] else 'PASSED'}")
md(f"  Score: {meta['score'].overall * 100:.0f}/100")


Raw prompt → BLOCKED

  Reason: Prompt quality 13/100 is below threshold 60/100


Templated prompt → PASSED

  Score: 84/100

## Step 4 — `OutputEvaluator`: score the output (7 dimensions)

If the **`Context`** carries **`constraints.forbidden_phrases`**, the evaluator **downgrades** outputs that include those strings — useful for catching boilerplate the prompt explicitly banned.

In [6]:
from mycontext.intelligence import OutputEvaluator

# Simulate an LLM output for scoring purposes
sample_output = """Key risks of deploying ML in regulated finance:

1. Model Risk (SR 11-7): Lack of documentation of model assumptions and limitations.
   Mitigation: Full model inventory, independent validation, shadow monitoring.

2. Data Risk: Training data may not reflect current market conditions post-crisis.
   Mitigation: Quarterly data freshness audits, concept drift detection.

3. Regulatory Risk: MiFID II and SR 11-7 require explainability for credit decisions.
   Mitigation: Use SHAP values; maintain audit-ready decision logs.

4. Operational Risk: Model failure during high-volatility periods.
   Mitigation: Circuit breakers, fallback to rules-based systems."""

evaluator = OutputEvaluator(mode="heuristic")
output_score = evaluator.evaluate(templated_ctx, sample_output)

md(f"Output quality: {output_score.overall * 100:.0f}/100")
md("\nDimension breakdown:")
for _dk, _dv in output_score.dimensions.items():
    _pct = _dv * 100.0
    _filled = int(_pct / 10)
    _bar = '█' * _filled + '░' * (10 - _filled)
    md(f"  {_dk.name.title():<28} {_bar} {_pct:.0f}/100")


Output quality: 44/100


Dimension breakdown:

  Instruction_Following        ██░░░░░░░░ 21/100

  Reasoning_Depth              ██░░░░░░░░ 29/100

  Actionability                ██░░░░░░░░ 22/100

  Structure_Compliance         ██████░░░░ 60/100

  Cognitive_Scaffolding        ███████░░░ 73/100

  Groundedness                 ██████░░░░ 60/100

  Register_Fit                 ██████░░░░ 65/100

## Step 5 — `ContextAmplificationIndex`: the template lift ratio

In [ ]:
from mycontext.intelligence import ContextAmplificationIndex

cai = ContextAmplificationIndex()
result = cai.measure(
    question=question,
    template_name="risk_assessor",
    provider="openai",
)

md(f"CAI (Context Amplification Index): {result.cai_overall:.2f}x")
md(f"Verdict: {result.verdict}")
md("\nPer-dimension lift:")
for dim_name, lift in result.cai_dimensions.items():
    direction = '↑' if lift > 1.0 else '↓'
    md(f"  {dim_name.name.title():<28} {direction} {lift:.2f}x")


## Step 6 — `TemplateBenchmark`: systematic testing

In [ ]:
from mycontext.intelligence import TemplateBenchmark

# List available benchmark definitions
bench = TemplateBenchmark()
available = bench.list_benchmarks()

md("Available benchmark definitions:")
for b in available:
    md(f"  • {b}")


In [ ]:
if available:
    benchmark_name = available[0]
    md(f"Running benchmark: {benchmark_name}")
    result = bench.run(benchmark_name, provider="openai")

    md(f"\nAverage CAI: {result.avg_cai:.2f}x")
    md(f"Cases run: {result.total_cases}")
    _pr = (result.passed / result.total_cases) if result.total_cases else 0.0
    md(f"Pass rate: {_pr:.0%}")
else:
    md("[Benchmark run requires OPENAI_API_KEY and benchmark YAML files]")


## Summary

| What you learned | API | Key? |
|-----------------|-----|------|
| Score prompt pre-execution (heuristic) | `QualityMetrics(mode='heuristic').evaluate(ctx)` | No |
| Score prompt pre-execution (LLM) | `QualityMetrics(mode='llm').evaluate(ctx)` | Yes |
| Quality gate pattern | `if score.overall < threshold: block` | No |
| Score output post-execution | `OutputEvaluator(mode='heuristic').evaluate(ctx, output)` | No |
| Measure template lift | `ContextAmplificationIndex().measure(question, template)` | Yes |
| Systematic benchmarking | `TemplateBenchmark().run(name, provider=...)` | Yes |

**Next notebook:** [05_multi_template_pipeline.ipynb](05_multi_template_pipeline.ipynb) — scale from one template to a full five-step reasoning pipeline, automatically suggested and integrated.